# 03 · Retrieval-augmented generation

**Session 01 · Building AI applications for research and data · notebook 3 of 6**

## Objective

Notebook 02 prepared the collection: it split the eight documents into section chunks, kept each document's date as metadata, and placed the chunks in a vector store. It ended on an open problem. A search for the latest figures put the 2016 regional summary first, and nothing used the date yet. This notebook uses the store to answer questions. For each question, the application searches the store for the chunks closest to it, places those chunks in the prompt as context, and the model writes the answer from them. A chunk that the search returns and the model reads is called a passage.

The pattern is called retrieval-augmented generation, or RAG. Retrieval is the search of the store, augmentation is the addition of the search results to the prompt, and generation is the model writing the answer. Notebook 02 built the first half of the pattern, the prepared collection. This notebook builds the second half, then shows three ways the pattern fails and the fix for each.

## The application after this notebook

Diagram: the question goes to retrieval, a search of the vector store that returns the closest chunks as passages; the passages and the question go into the prompt, then to generation, where the model writes an answer with citations.

Retrieval and the returned passages are the parts this notebook adds, and the model's step becomes generation. Together they form the RAG pattern.

## Contents

1. [Section 1](#scrollTo=section-1) answers the main question from the store. The answer cites its passages and says that half of the question cannot be answered from them.
2. [Section 2](#scrollTo=section-2) shows the ranking behind that answer. The economy chunks sit below the cut-off, so the model never saw them. Searching once for each part of the question brings them back.
3. [Section 3](#scrollTo=section-3) asks for the latest figures and shows the 2016 summary outranking the 2026 briefs. The model reports a 2015 value because that is what it was given.
4. [Section 4](#scrollTo=section-4) tries two fixes for outdated evidence. Rewording the question does not help, and a filter on the date kept in notebook 02 removes the summary.
5. [Section 5](#scrollTo=section-5) asks two questions the collection cannot answer and shows that the search returns passages anyway.
6. [Section 6](#scrollTo=section-6) tests the search on the eight evaluation questions without calling the model.

## How to run it

Run the cells in order. Without an API key, the model's answers are recorded ones, labelled `[recorded output: ...]`, as in notebook 01. With a key in `WORKSHOP_API_KEY`, the same cells call the model live, and the answers will differ in wording from the recorded ones. The searches, the rankings, and the tables are computed when a cell runs, in both modes.

The next cell installs the session's tools and loads its helper module. The first run takes about a minute. The cell is setup only.

In [1]:
# Setup. In Colab this clones the repository and installs the dependencies; locally it uses the folder as is.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/worldbank/ai4data.git"   # the program's repository
BRANCH = "main"   # the branch the Colab setup clones
SESSION = "training/session-01-building-ai-applications/notebooks"

if not pathlib.Path("workshop.py").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/ai4data"], check=True)
    os.chdir(f"/content/ai4data/{SESSION}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, os.getcwd())

# Optional: a key for an OpenAI-compatible API. Without one, the notebook replays recorded outputs and says so.
try:
    from google.colab import userdata  # type: ignore
    for name in ("WORKSHOP_API_KEY", "WORKSHOP_BASE_URL", "WORKSHOP_MODEL"):
        if userdata.get(name):
            os.environ[name] = userdata.get(name)
except Exception:
    pass

import workshop as w
model = w.make_model()
print("Model:", w.mode(model))

Model: recorded outputs (no WORKSHOP_API_KEY set)


The next cell builds the same store as notebook 02 in three lines. A Colab notebook starts with an empty session, so the store from notebook 02 is not available here, and the three lines rebuild it from the documents. The cell also sets up the application: the instructions, a prompt with a slot for the passages, and two functions. The first function searches the store and asks the model. The second only searches, and returns the chunks with their scores. The number `k` is how many passages a search returns, and the model reads only those.

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

docs = w.load_documents()
chunks = w.split_sections(docs)
store = w.build_store(chunks)
print(len(chunks), "chunks in the store, the same collection as in notebook 02")

INSTRUCTIONS = """You are a research assistant for development data.
Answer only from the supplied passages. After each figure, cite the passage in the square-bracket form given.
If the passages do not contain what is needed, say so."""

prompt = ChatPromptTemplate.from_messages([("system", INSTRUCTIONS), ("human", "PASSAGES:\n{context}\n\nQUESTION: {question}")])


def make_rag(k=4, where=None):
    """RAG (retrieval-augmented generation): search the store, put the passages in the prompt, ask the model."""
    retrieve = RunnableLambda(lambda q: store.similarity_search(q, k=k, filter=where))
    return {"context": retrieve | w.format_context, "question": RunnablePassthrough()} | prompt | model


def hits(question, k=4, where=None):
    return store.similarity_search_with_score(question, k=k, filter=where)

21 chunks in the store, the same collection as in notebook 02


## 1. The main question answered from the store

The question is the main one from notebook 01. The cell prints the question, then the four chunks closest to it with their similarity scores, then the model's answer written from those four passages. A score is the cosine similarity between the question and a chunk, and a higher score means a closer match.

In [3]:
w.show(w.MAIN_QUESTION, role="user")
print()
w.print_hits(hits(w.MAIN_QUESTION, k=4))
print()
w.use("nb03_rag_main")
w.show(make_rag(k=4).invoke(w.MAIN_QUESTION))


 0.763  [rwanda-digital · Internet use · 2026]
 0.748  [regional-2016 · intro · 2016]
 0.676  [kenya-digital · Internet use · 2026]
 0.591  [mobile-money · intro · 2026]



The answer cites a passage after every figure, and it opens by saying what it cannot do: the passages contain no economic indicators, so it covers internet access only. That is the correct behaviour under these instructions, and its figures are right. Half of the question still went unanswered, although the briefs with the GDP figures are in the store. Section 2 shows why they were not returned.

## 2. The ranking behind the answer

The question asks about internet access and economic development together. The search compares the whole question with every chunk and keeps the four closest. The cell ranks the ten closest chunks for the question, colours each one by topic, and draws the cut-off line after the fourth, which is where the model stopped seeing them.

In [4]:
topic = {"kenya-digital": "internet", "rwanda-digital": "internet", "regional-2016": "internet", "internet-indicator": "internet",
         "kenya-economy": "economy", "rwanda-economy": "economy", "coffee": "other", "mobile-money": "other"}
legend = {"internet": ("internet use", "#2f5f9e"), "economy": ("economy", "#d9822b"), "other": ("other topics", "#8a94a3")}

ranked = hits(w.MAIN_QUESTION, k=10)
w.show_ranking([(w.citation(d), score, topic[d.metadata["id"]]) for d, score in ranked], k=4, legend=legend, question=w.MAIN_QUESTION)

The four chunks above the line are three internet chunks, one of them the 2016 summary, and the mobile-money note. The first economy chunk is fifth at 0.570, against 0.591 for the fourth, and the chunk with Kenya's GDP per capita is ninth. The search has no notion that the question has two halves. It ranks every chunk by its closeness to the whole question, and the internet half dominates the match. Returning more passages is a poor remedy: Kenya's GDP per capita appears only when nine passages are returned, the coffee note comes along in seventh place, and every passage adds tokens and cost, as notebook 01 showed.

The fix acts on the search. The question is broken into its two parts, internet use and the economy, each part is searched once, and the passages are merged. Libraries can have a model write the smaller questions; here they are written by hand so that the mechanism is visible. The cell shows the single search and the merged search side by side and marks which of the four figures the question needs each one brings back. The merged search has a cost. Two searches of four chunks each give eight different passages, against four before, and the 2016 summary and the mobile-money note come along with them.

In [5]:
# the question broken into its two parts: one smaller question for each
sub_queries = ["internet use in Kenya and Rwanda, recent figures", "GDP per capita and economic growth in Kenya and Rwanda, recent figures"]
merged = []
for q in sub_queries:
    for d in store.similarity_search(q, k=4):
        if d not in merged:
            merged.append(d)

one_search = [d for d, _ in hits(w.MAIN_QUESTION, k=4)]
needed = next(item for item in w.load_questions() if item["id"] == "q1")["expected_numbers"]
countries = {"KEN": "Kenya", "RWA": "Rwanda"}
names = {"IT.NET.USER.ZS": "internet use", "NY.GDP.PCAP.CD": "GDP per capita"}
checks = [f"{countries[n['country']]} {names[n['indicator']]}, {n['year']}: {n['value']:,}" for n in needed]


def column(title, tone, found):
    text = " ".join(d.page_content for d in found)
    return {"title": title, "tone": tone, "hits": [(w.citation(d), d.page_content) for d in found],
            "found": [w.numbers_present(text, [n])[0] == 1 for n in needed]}


w.show_search_comparison(w.MAIN_QUESTION, [column("One search for the whole question: 4 passages", "bad", one_search),
                                           column(f"The question broken into two, one search each, merged: {len(merged)} passages", "good", merged)], checks)
print()
w.use("nb03_rag_two_queries")
w.show((prompt | model).invoke({"context": w.format_context(merged), "question": w.MAIN_QUESTION}))

The merged passages contain all four figures, where the single search contained two, and the answer covers both halves of the question with a citation after every figure. The two extra passages were used as well. The model drew on the 2016 summary for its projection and noted that the projection cannot be checked against the supplied data, and it reported that the mobile-money note holds no figures. Section 3 shows what happens when the 2016 summary is the passage an answer depends on.

## 3. Outdated evidence in the ranking

Notebook 02 ended with the 2016 summary ranking first for a question about the latest figures. This section shows the effect of that ranking on an answer. The question is "What are the latest available figures for internet use in Kenya and Rwanda?", and the 2016 summary contains the words "latest available figures". The search returns two passages. The cell prints them with their scores, then the answer.

In [6]:
q = "What are the latest available figures for internet use in Kenya and Rwanda?"
w.print_hits(hits(q, k=2))
print()
w.use("nb03_rag_outdated")
w.show(make_rag(k=2).invoke(q))

 0.816  [regional-2016 · intro · 2016]
 0.765  [rwanda-digital · Internet use · 2026]



The 2016 summary ranks first at 0.816 and Rwanda's 2026 brief second at 0.765. Kenya's brief is not among the two, so the model's Kenya figure comes from the summary: 16.6 percent in 2015. The model is careful about it. It says that the summary dates from 2016, that the passages hold no later Kenya figure, and that 16.6 is the most recent value it can report. Rwanda's figure, 31.7 percent in 2024, comes from the brief. The answer is faithful to its passages and still wrong for the reader, who asked for the latest figures while the 2024 value for Kenya sits in the store. The generation step cannot repair this, because the passages were selected before the model saw them. The fix has to act on the search.

## 4. Three fixes for outdated evidence

Three fixes act on the search side.

1. **A filter on the date.** The search ignores documents dated before 2020. Notebook 02 kept each document's date as metadata for this purpose.
2. **A reworded question.** The question is rewritten to resemble the expected passage, for example "internet use 2024, percent of population". This often helps, and the cells below show that it does not help here.
3. **Sorting by date after the search**, for a store without a filter. It does the same job as the filter and is not shown.

The first cell compares the same search without and with the date filter, and marks which of the two figures the question needs turn up.

In [7]:
since_2020 = lambda d: str(d.metadata.get("date", "")) >= "2020"
expected = next(item for item in w.load_questions() if item["id"] == "q8")["expected_numbers"]
countries = {"KEN": "Kenya", "RWA": "Rwanda"}
checks = [f"{countries[e['country']]}, {e['year']}: {e['value']}" for e in expected]


def search(title, tone, where=None):
    found = store.similarity_search(q, k=2, filter=where)
    text = " ".join(h.page_content for h in found)
    return {"title": title, "tone": tone, "hits": [(w.citation(h), h.page_content) for h in found],
            "found": [w.numbers_present(text, [e])[0] == 1 for e in expected]}


w.show_search_comparison(q, [search("No filter", "bad"), search("Date filter: 2020 or later", "good", since_2020)], checks)

With the filter, the 2016 summary is gone, Kenya's brief takes its place, and both 2024 figures are present. The next cell shows the model's answer from the filtered passages. The cell after it tries the reworded question without the filter.

In [8]:
w.use("nb03_rag_filtered")
w.show(make_rag(k=2, where=since_2020).invoke(q))

In [9]:
w.print_hits(hits("Kenya and Rwanda internet use 2024, percent of population", k=2))

 0.871  [regional-2016 · intro · 2016]
 0.833  [rwanda-digital · Internet use · 2026]


The reworded question still puts the 2016 summary first, at 0.871, above Rwanda's brief at 0.833. The summary uses the same vocabulary as the question, so rewording strengthens the match. The date is the only property that separates the summary from the briefs, and the search can use it only because notebook 02 kept it.

## 5. A question outside the collection

A search always returns the closest chunks, even when none of them is close. The cell asks two questions: one the collection can answer, the population of Rwanda, and one it cannot, inflation in Uganda in 2023. Each gets three passages with their scores.

In [10]:
for q2 in ["What is the population of Rwanda?", "What was inflation in Uganda in 2023?"]:
    print(q2)
    w.print_hits(hits(q2, k=3))
    print()

What is the population of Rwanda?
 0.739  [rwanda-digital · Population · 2026]
 0.598  [rwanda-economy · Reading the two indicators together · 2026]
 0.595  [rwanda-economy · GDP per capita · 2026]

What was inflation in Uganda in 2023?
 0.534  [rwanda-economy · GDP per capita · 2026]
 0.523  [kenya-economy · GDP per capita · 2026]
 0.506  [rwanda-economy · Reading the two indicators together · 2026]



Both questions get three passages. For the population question the right chunk comes first at 0.739. For the inflation question the best score is 0.534, and all three passages are about GDP per capita in Kenya and Rwanda, which say nothing about Uganda. A cut-off score would separate these two cases, and the cut-off has to be set from examples, because it depends on the embedding model in use. The second safeguard is the instruction from notebook 01 that tells the model to say when the passages do not answer the question. Notebook 05 shows that instruction at work. Applications use both.

## 6. A test of the search without the model

The search can be tested before any model is involved, because retrieval is the first half of RAG and the model is the second. For each of the eight evaluation questions, the cell makes two checks: whether the passages include at least one of the expected documents, and how many of the numbers a good answer needs appear in the passages. `K` is the number of passages the search returns.

In [11]:
K = 4   # passages returned by the search: try 2, then 8

rows = []
for item in w.load_questions():
    passages = store.similarity_search(item["question"], k=K)
    text = " ".join(p.page_content for p in passages)
    needed = item.get("expected_numbers", [])
    found = sum(w.numbers_present(text, [n])[0] for n in needed)
    rows.append((item["id"], item["category"],
                 "hit" if w.retrieval_hit(passages, item["expected_documents"]) else "miss",
                 f"{found}/{len(needed)}" if needed else "none needed"))
w.show_table(["id", "category", "expected document", "needed numbers"], rows, source=f"The search returns {K} passages per question.")

id,category,expected document,needed numbers
q1,main,hit,2/4
q2,factual,hit,1/1
q3,ambiguous,hit,none needed
q4,missing-data,hit,none needed
q5,out-of-scope,hit,none needed
q6,factual,hit,2/2
q7,definition,hit,none needed
q8,outdated-evidence,hit,2/2


At `K = 4`, every question passes the first check, and the main question q1 fails the second: its passages hold two of the four numbers the answer needs, the two internet figures, which is the problem section 2 found. That is a retrieval problem, and no instruction to the model can fix it. When both checks pass and an answer is still wrong, the fault is in the generation step, the instructions or the model, which notebook 06 measures. Testing the two halves separately shows which one to change.

## Key points

- Retrieval-augmented generation searches the store for the chunks closest to the question, places them in the prompt as passages, and has the model answer from them with a citation after each figure.
- The search decides what the answer can say. For the main question the economy chunks ranked below the cut-off and never reached the model. Searching once for each part of the question brought them back.
- A similarity search ranks by meaning and ignores the date. The 2016 summary outranked the 2026 briefs, rewording the question strengthened the match, and a filter on the date kept as metadata removed the summary.
- A search always returns something, even for a question the collection cannot answer. Two safeguards apply: a cut-off score set from examples, and an instruction that tells the model to say when the passages do not answer.
- The search can be tested without the model, by checking that the expected documents and the needed numbers are among the passages. A miss there cannot be fixed by changing the instructions.

## Exercise

1. Look at the ranking in section 2. Which chunk takes fourth place, and is it a good use of one of the four slots? Say what in the chunk explains its score.
2. In the cell of section 6, set `K` to `2`, run it, then set it to `8` and run it again. Which questions lose numbers at 2, and which gain at 8? At 8, does every question find all of its needed numbers?
3. Set `K` to `2` and look at q8, the question from section 3. Explain why it passes at 4 and fails at 2, using the two searches in section 4.

## Where this connects to the program

- [Practical Guide to Serving Official Statistics to AI Agents](https://worldbank.github.io/ai4data/cookbook/serving-statistics-to-agents/): the design of tools over a statistical catalog, in full.
- [Practical Guide to Evaluation Suites for Statistical AI](https://worldbank.github.io/ai4data/cookbook/evaluation-suites/): how to build and run the tests.
- [Proof-Carrying Numbers](https://worldbank.github.io/ai4data/docs/pcn/): the claim-tag protocol the verifier in this session follows. See also the [paper](https://arxiv.org/abs/2509.06902) and the [`pcn` repository](https://github.com/worldbank/pcn).
- [Data360 MCP server](https://github.com/worldbank/data360-mcp): the tools of notebook 04, exposed to any MCP client over the World Bank's data platform.
- [End-to-end reference implementation](https://worldbank.github.io/ai4data/docs/reference-implementation/): documentation, validation, publication, agent access, and verification in one run.

**Next:** [04 · Tools and structured outputs](https://colab.research.google.com/github/worldbank/ai4data/blob/main/training/session-01-building-ai-applications/notebooks/04-tools-and-structured-outputs.ipynb)